In [1]:
import pandas as pd
import numpy as np
import json
from sktime.performance_metrics.forecasting import mean_absolute_scaled_error, mean_absolute_percentage_error
import statistics
import plotly.express as px

In [2]:
def calculate_error(error_type, observed_values, forecasts, training_data):
    if error_type == 'MASE':
        error = mean_absolute_scaled_error(observed_values, forecasts, y_train = training_data)
    elif error_type == 'MAPE':
        error = mean_absolute_percentage_error(observed_values, forecasts)
    error = float('%.2f' % round(error, 2))
    return error

In [3]:
exp_types = ['fc', 'gc-fc', 'cp-gc-fc']
gc_exp_types = ['gc-fc', 'cp-gc-fc']
logs = ['p2p', 'order-management', 'sim-order-management']
log_error_dict = {'MASE': {'order-management': {'fc': {}, 'gc-fc': {}, 'cp-gc-fc': {}}, \
                  'p2p': {'fc': {}, 'gc-fc': {}, 'cp-gc-fc': {}}, \
                    'sim-order-management': {'fc': {}, 'gc-fc': {}, 'cp-gc-fc': {}, 'context-based-pp2-so': {}, 'context-based-pp2-so-cp': {}, 'context-based-pp6-so': {}, 'context-based-pp6-so-cp': {}}},
                    'MAPE': {'order-management': {'fc': {}, 'gc-fc': {}, 'cp-gc-fc': {}}, \
                    'p2p': {'fc': {}, 'gc-fc': {}, 'cp-gc-fc': {}}, \
                    'sim-order-management': {'fc': {}, 'gc-fc': {}, 'cp-gc-fc': {}, 'context-based-pp2-so': {}, 'context-based-pp2-so-cp': {}, 'context-based-pp6-so': {}, 'context-based-pp6-so-cp': {}}}}
log_num_of_vars_dict = {'order-management': {'gc-fc': {}, 'cp-gc-fc': {}}, \
                  'p2p': {'gc-fc': {}, 'cp-gc-fc': {}}, \
                    'sim-order-management': {'gc-fc': {}, 'cp-gc-fc': {}}}
error_types = ['MASE', 'MAPE']

Calculate Errors

In [4]:
tsid_list = []
for error_type in error_types:
    for log in logs:
        for exp_type in exp_types:
            with open(f'{log}\\{exp_type}\\Forecasting_results.json') as handle:
                ar_collection = json.loads(handle.read())
            error_dict = {}
            for ar in ar_collection:
                ar_tsid = ar['tsid']
                forecasts = ar['ar']
                forecasts_df = pd.DataFrame.from_dict(forecasts, orient='index', columns = ['forecasts'])
                forecasts_df.index = pd.to_datetime(forecasts_df.index, utc= True)
                tsid = str(ar_tsid).replace('[', '(').replace(']',')')
                ts_orig = pd.read_csv(f'{log}\\ts\\{tsid}.csv', index_col=0)
                ts_orig.index = pd.to_datetime(ts_orig.index, utc= True)
                ts_train = ts_orig.tail(-4)
                ts_true = ts_orig.tail(4)
                error = calculate_error(error_type, ts_true, forecasts_df, ts_train)
                if (ts_true < 0.001).any().values[0]:
                    continue
                error_dict[tsid] = error
                tsid_list.append(tsid)
            log_error_dict[error_type][log][exp_type] = error_dict

In [5]:
tsid_list = []
log = 'sim-order-management'
cb_exp_types = ['context-based-pp2-so', 'context-based-pp2-so-cp', 'context-based-pp6-so', 'context-based-pp6-so-cp']

for error_type in error_types:
    for exp_type in cb_exp_types:
        with open(f'{log}\\{exp_type}\\Forecasting_results.json') as handle:
            ar_collection = json.loads(handle.read())
        error_dict = {}
        for ar in ar_collection:
            ar_tsid = ar['tsid']
            forecasts = ar['ar']
            forecasts_df = pd.DataFrame.from_dict(forecasts, orient='index', columns = ['forecasts'])
            forecasts_df.index = pd.to_datetime(forecasts_df.index, utc= True)
            tsid = str(ar_tsid).replace('[', '(').replace(']',')')
            ts_orig = pd.read_csv(f'{log}\\ts\\{tsid}.csv', index_col=0)
            ts_orig.index = pd.to_datetime(ts_orig.index, utc= True)
            ts_train = ts_orig.tail(-4)
            ts_true = ts_orig.tail(4)
            error = calculate_error(error_type, ts_true, forecasts_df, ts_train)
            if (ts_true < 0.001).any().values[0]:
                continue
            error_dict[tsid] = error
            tsid_list.append(tsid)
        log_error_dict[error_type][log][exp_type] = error_dict


Get Number of Exogenous Variables

In [6]:
for log in logs:
    for exp_type in gc_exp_types:
        with open(f'{log}\\{exp_type}\\Granger Causality_results.json') as handle:
            ar_collection = json.loads(handle.read())
        num_of_vars_dict = {}
        for ar in ar_collection:
            ar_tsid = ar['tsid']
            causal_variables = ar['ar']
            tsid = str(ar_tsid).replace('[', '(').replace(']',')')
            if tsid in log_error_dict[error_types[0]][log][exp_type].keys():
                num_of_vars_dict[tsid] = len(causal_variables)
        log_num_of_vars_dict[log][exp_type] = num_of_vars_dict

In [7]:
tsid_list = []
log = 'sim-order-management'
cb_exp_types = ['context-based-pp2-so', 'context-based-pp2-so-cp', 'context-based-pp6-so', 'context-based-pp6-so-cp']

for error_type in error_types:
    for exp_type in cb_exp_types:
        with open(f'{log}\\{exp_type}\\Granger Causality_results.json') as handle:
            ar_collection = json.loads(handle.read())
        num_of_vars_dict = {}
        for ar in ar_collection:
            ar_tsid = ar['tsid']
            causal_variables = ar['ar']
            tsid = str(ar_tsid).replace('[', '(').replace(']',')')
            if tsid in log_error_dict[error_types[0]][log][exp_type].keys():
                num_of_vars_dict[tsid] = len(causal_variables)
        log_num_of_vars_dict[log][exp_type] = num_of_vars_dict

In [8]:
for log in logs:
    for exp_type in gc_exp_types:
        gc_keys = [*log_num_of_vars_dict[log][exp_type].keys()]
        fc_keys = [*log_error_dict[error_types[0]][log][exp_type].keys()]
        no_exo_keys = list(set(fc_keys) - set(gc_keys))
        for key in no_exo_keys:
            log_num_of_vars_dict[log][exp_type][key] = 0

Create dataframes for errors and no. of exogenous variables to pass as input to plotly

In [9]:
metrics_df_dict = {}
for error_type in error_types:
    row_list = []
    for log in logs:
        for exp in exp_types:
            values = [*log_error_dict[error_type][log][exp].values()]
            for value in values:
                if log == 'order-management':
                    log_abbr = 'OM'
                elif log == 'p2p':
                    log_abbr = 'P2P'
                elif log == 'sim-order-management':
                    log_abbr = 'SOM'
                value_dict = {}
                value_dict = {'log':log_abbr, 'experiment':exp, 'error': value}
                row_list.append(value_dict)
    metrics_df_dict[error_type] = pd.DataFrame(row_list)

In [10]:
row_list = []
for log in logs:
    for exp in gc_exp_types:
        values = [*log_num_of_vars_dict[log][exp].values()]
        for value in values:
            if log == 'order-management':
                log_abbr = 'OM'
            elif log == 'p2p':
                log_abbr = 'P2P'
            elif log == 'sim-order-management':
                log_abbr = 'SOM'
            value_dict = {}
            value_dict = {'log':log_abbr, 'experiment':exp, '# of exog. vars': value}
            row_list.append(value_dict)
num_of_vars_df= pd.DataFrame(row_list)

Calculate mean error and mean/median no. of exogenous variables

In [11]:
for error_type in error_types:
    for log in logs:
        for exp_type in exp_types:
            error_mean = statistics.mean([*log_error_dict[error_type][log][exp_type].values()])
            error_median = statistics.median([*log_error_dict[error_type][log][exp_type].values()])
            error_mean = float('%.2f' % round(error_mean, 2))
            error_median = float('%.2f' % round(error_median, 2))
            print(f'error_type: {error_type}, log: {log}, exp: {exp_type}, error_mean: {error_mean}, error_median: {error_median} \n')
        print('\n\n')

error_type: MASE, log: p2p, exp: fc, error_mean: 0.83, error_median: 0.76 

error_type: MASE, log: p2p, exp: gc-fc, error_mean: 6.74, error_median: 0.77 

error_type: MASE, log: p2p, exp: cp-gc-fc, error_mean: 0.84, error_median: 0.79 




error_type: MASE, log: order-management, exp: fc, error_mean: 0.8, error_median: 0.67 

error_type: MASE, log: order-management, exp: gc-fc, error_mean: 3.91, error_median: 0.74 

error_type: MASE, log: order-management, exp: cp-gc-fc, error_mean: 1.32, error_median: 0.71 




error_type: MASE, log: sim-order-management, exp: fc, error_mean: 1.82, error_median: 1.48 

error_type: MASE, log: sim-order-management, exp: gc-fc, error_mean: 20.77, error_median: 1.76 

error_type: MASE, log: sim-order-management, exp: cp-gc-fc, error_mean: 2.04, error_median: 1.61 




error_type: MAPE, log: p2p, exp: fc, error_mean: 0.28, error_median: 0.22 

error_type: MAPE, log: p2p, exp: gc-fc, error_mean: 3.09, error_median: 0.21 

error_type: MAPE, log: p2p, exp: cp

In [12]:
for log in logs:
    for exp_type in gc_exp_types:
        num_of_vars_mean = statistics.mean([*log_num_of_vars_dict[log][exp_type].values()])
        num_of_vars_median = statistics.median([*log_num_of_vars_dict[log][exp_type].values()])
        print(f'log: {log}, exp: {exp_type}, num_of_vars_mean: {num_of_vars_mean}, num_of_vars_median: {num_of_vars_median} \n')
    print('\n\n')

log: p2p, exp: gc-fc, num_of_vars_mean: 6.337078651685394, num_of_vars_median: 5 

log: p2p, exp: cp-gc-fc, num_of_vars_mean: 1.2247191011235956, num_of_vars_median: 0 




log: order-management, exp: gc-fc, num_of_vars_mean: 11.067567567567568, num_of_vars_median: 10.0 

log: order-management, exp: cp-gc-fc, num_of_vars_mean: 2.7432432432432434, num_of_vars_median: 0.0 




log: sim-order-management, exp: gc-fc, num_of_vars_mean: 3.75, num_of_vars_median: 4.0 

log: sim-order-management, exp: cp-gc-fc, num_of_vars_mean: 1.85, num_of_vars_median: 2.0 






Generate plots (Note: First two plots have been rescaled and not all outliers are visible by default. We further remove even those that are visible before using the plots in the written report.)

In [13]:
df = metrics_df_dict['MASE']

fig = px.box(df, x="log", y="error", color="experiment")
fig.update_layout( 
    yaxis=dict(
        range=[0, 5]
    ),
legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.01
)
)
fig.show()

c:\Users\Ibrahim\anaconda3\envs\Measuring_OCED\Lib\site-packages\kaleido\_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




In [14]:
df = metrics_df_dict['MAPE']

fig = px.box(df, x="log", y="error", color="experiment")
fig.update_layout( 
    yaxis=dict(
        range=[0, 1]
    ),
legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.01
)
)
fig.show()

In [15]:
df = num_of_vars_df

fig = px.box(df, x="log", y="# of exog. vars", color="experiment", color_discrete_sequence=[ "#EF553B", "#00CC96"])
fig.update_layout(legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="left",
    x=0.01
))

fig.show()

Get no. of exogenous variables and errors for ('pp2', 'ship order')

In [16]:
log = 'sim-order-management'
ts_key = "('pp2', 'ship order')"
exp_types_mod = exp_types + ['context-based-pp2-so', 'context-based-pp2-so-cp']
for error_type in error_types:
    for exp_type in exp_types_mod:
        if exp_type == 'fc':
            num_of_exog_vars = 0
        else:
            num_of_exog_vars = log_num_of_vars_dict[log][exp_type][ts_key]
        print(f'no. of exogenous variables: {num_of_exog_vars}, error type: {error_type},  experiment: {exp_type}, value: {log_error_dict[error_type][log][exp_type][ts_key]}')

no. of exogenous variables: 0, error type: MASE,  experiment: fc, value: 2.03
no. of exogenous variables: 7, error type: MASE,  experiment: gc-fc, value: 3.41
no. of exogenous variables: 2, error type: MASE,  experiment: cp-gc-fc, value: 3.68
no. of exogenous variables: 5, error type: MASE,  experiment: context-based-pp2-so, value: 1.64
no. of exogenous variables: 1, error type: MASE,  experiment: context-based-pp2-so-cp, value: 0.79
no. of exogenous variables: 0, error type: MAPE,  experiment: fc, value: 0.05
no. of exogenous variables: 7, error type: MAPE,  experiment: gc-fc, value: 0.09
no. of exogenous variables: 2, error type: MAPE,  experiment: cp-gc-fc, value: 0.1
no. of exogenous variables: 5, error type: MAPE,  experiment: context-based-pp2-so, value: 0.05
no. of exogenous variables: 1, error type: MAPE,  experiment: context-based-pp2-so-cp, value: 0.02


Get no. of exogenous variables and errors for ('pp6', ('ship order', 'items'))

In [17]:
log = 'sim-order-management'
ts_key = "('pp6', ('ship order', 'items'))"
exp_types_mod = exp_types + ['context-based-pp6-so', 'context-based-pp6-so-cp']
for error_type in error_types:
    for exp_type in exp_types_mod:
        if exp_type == 'fc':
            num_of_exog_vars = 0
        else:
            num_of_exog_vars = log_num_of_vars_dict[log][exp_type][ts_key]
        print(f'no. of exogenous variables: {num_of_exog_vars}, error type: {error_type},  experiment: {exp_type}, value: {log_error_dict[error_type][log][exp_type][ts_key]}')

no. of exogenous variables: 0, error type: MASE,  experiment: fc, value: 3.63
no. of exogenous variables: 9, error type: MASE,  experiment: gc-fc, value: 358.28
no. of exogenous variables: 4, error type: MASE,  experiment: cp-gc-fc, value: 1.64
no. of exogenous variables: 5, error type: MASE,  experiment: context-based-pp6-so, value: 1.88
no. of exogenous variables: 1, error type: MASE,  experiment: context-based-pp6-so-cp, value: 0.91
no. of exogenous variables: 0, error type: MAPE,  experiment: fc, value: 0.1
no. of exogenous variables: 9, error type: MAPE,  experiment: gc-fc, value: 9.98
no. of exogenous variables: 4, error type: MAPE,  experiment: cp-gc-fc, value: 0.04
no. of exogenous variables: 5, error type: MAPE,  experiment: context-based-pp6-so, value: 0.05
no. of exogenous variables: 1, error type: MAPE,  experiment: context-based-pp6-so-cp, value: 0.03


Get no. of exogenous variables and errors for ('ep1', 'place order')

In [18]:
log = 'sim-order-management'
ts_key = "('ep1', 'place order')"
for error_type in error_types:
    for exp_type in exp_types:
        if exp_type == 'fc':
            num_of_exog_vars = 0
        else:
            num_of_exog_vars = log_num_of_vars_dict[log][exp_type][ts_key]
        print(f'no. of exogenous variables: {num_of_exog_vars}, error type: {error_type},  experiment: {exp_type}, value: {log_error_dict[error_type][log][exp_type][ts_key]}')

no. of exogenous variables: 0, error type: MASE,  experiment: fc, value: 1.31
no. of exogenous variables: 0, error type: MASE,  experiment: gc-fc, value: 1.31
no. of exogenous variables: 0, error type: MASE,  experiment: cp-gc-fc, value: 1.31
no. of exogenous variables: 0, error type: MAPE,  experiment: fc, value: 0.03
no. of exogenous variables: 0, error type: MAPE,  experiment: gc-fc, value: 0.03
no. of exogenous variables: 0, error type: MAPE,  experiment: cp-gc-fc, value: 0.03
